## Imports

In [ ]:
import warnings

warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

In [ ]:
from operator import itemgetter

import networkx as nx
import numpy as np
from astropy import units as u
from astropy.coordinates import ICRS, EarthLocation, SkyCoord
from astropy.table import QTable
from astropy.time import Time
from astropy_healpix import HEALPix
from ligo.skymap import plot
from m4opt import fov
from m4opt.dynamics import nominal_roll
from m4opt.fov._core import circle_to_polygon
from m4opt.missions import uvex as mission
from m4opt.skygrid import _geodesic
from m4opt.utils.optimization import partition_graph, partition_graph_color, solve_tsp
from matplotlib import cm, colors
from matplotlib import pyplot as plt
from regions import CircleSkyRegion, PointSkyRegion, Regions

Generate sky grid.

In [ ]:
geodesic_args = (21, 4, "icosahedron")
coords = _geodesic.for_subdivision(*geodesic_args)
assert (coords == mission.skygrid).all()
fields = QTable({"field_id": np.arange(len(coords)), "target_coord": coords})
fields

Determine the fraction of the area of each field that overlaps with each survey region.

In [ ]:
# Read region files
(inscribed_circle,) = Regions.read("../fov-inscribed-circle.ds9")
lmlz_deep = Regions.read("../survey-footprints/lmlz-deep.ds9")
regions = {
    "lmlz_wide": Regions.read("../survey-footprints/lmlz-wide.ds9"),
    "lmlz_deep": Regions(
        [region for region in lmlz_deep if not isinstance(region, PointSkyRegion)]
    ),
    "lmlz_deep_points": Regions(
        [region for region in lmlz_deep if isinstance(region, PointSkyRegion)]
    ),
    "mc": Regions.read("../survey-footprints/magellanic-clouds.ds9"),
}

# Regions for plots
plot_regions = [
    circle_to_polygon(region, 8) if isinstance(region, CircleSkyRegion) else region
    for value in regions.values()
    for region in value
    if not isinstance(region, PointSkyRegion)
]

hpx = HEALPix(512, frame=ICRS())
hpx_fields = [
    set(ipix) for ipix in fov.footprint_healpix(hpx, inscribed_circle, coords)
]

for key, value in regions.items():
    hpx_regions = set(fov.footprint_healpix(hpx, value))
    fields[f"overlap_{key}"] = [
        len(hpx_regions.intersection(hpx_field)) / len(hpx_field)
        for hpx_field in hpx_fields
    ]
fields

Special case for the Magellanic Clouds, which are a non-contiguous region. We rank the fields by decreasing overlap with the Magellanic Cloud regions, and then select the largest number of fields that we can visit in a single scheduling block, accounting for slews.

In [ ]:
exptime_per_field = 900 * u.s
worst_case_slew_time = mission.slew.time(
    SkyCoord(0 * u.deg, 0 * u.deg), SkyCoord(180 * u.deg, 0 * u.deg)
)
downlink_time = 1800 * u.s
block_duration = 6 * u.hour
available_time = block_duration - downlink_time - 2 * worst_case_slew_time

i = np.argsort(fields["overlap_mc"])[:-21:-1]

roll = nominal_roll(
    EarthLocation.from_geocentric(0, 0, 0, u.m),
    fields["target_coord"][i],
    Time("2025-01-01"),
)
slew_time = mission.slew.time(
    fields["target_coord"][i],
    fields["target_coord"][i, np.newaxis],
    roll,
    roll[:, np.newaxis],
).to_value(u.s)

for nfields in range(len(i), 1, -1):
    _, objective = solve_tsp(slew_time[:nfields, :nfields], verbose=False)
    total_time = objective * u.s + exptime_per_field * nfields
    if total_time <= available_time:
        break
else:
    raise RuntimeError("No solution found")

in_mc = np.zeros(len(fields), dtype=bool)
in_mc[i[:nfields]] = True
fields["in_mc"] = in_mc

fields

In [ ]:
overlap_fraction_threshold = 0.25
fields["in_lmlz_deep_points"] = fields["overlap_lmlz_deep_points"] > 0
fields["in_lmlz_deep"] = fields["overlap_lmlz_deep"] >= overlap_fraction_threshold
fields["in_lmlz_wide"] = fields["overlap_lmlz_wide"] >= overlap_fraction_threshold
fields

Plot number of exposures.

In [ ]:
# Exposure times and candeces taken from UVEX Design Reference Mission
# Version 1.0, 11/20/2025
base_limmag = np.asarray([25.25, 24.25])  # All-sky (NUV, MUV) survey depth
allsky_visits = 3
allsky_limmag = base_limmag + 1.25 * np.log10(
    allsky_visits
)  # 3 exposures per year, cadence of 1/(4 months)

lmlz_wide_limmag = np.asarray([25.75, 25.75])
lmlz_deep_limmag = np.asarray([27.0, 27.0])
mc_limmag = np.asarray([24.5, 24.5])  # Magellanic clouds survey depth
mc_limmag += 1.25 * np.log10(52 * 2)  # 2/week cadence

region_names = ["lmlz_wide", "lmlz_deep", "lmlz_deep_points", "mc"]
region_costs = 10 ** (
    (
        np.asarray([lmlz_wide_limmag, lmlz_deep_limmag, lmlz_deep_limmag, mc_limmag])
        - allsky_limmag
    ).max(axis=-1)
    / 1.25
)

costs = np.full(len(coords), 1)
for region_cost, region_name in sorted(zip(region_costs, region_names)):
    costs[fields[f"in_{region_name}"]] = region_cost

fig = plt.figure()
shift = 4 * u.hourangle
ax = plt.axes(
    projection="astro aitoff", center=SkyCoord(12 * u.hourangle - shift, 0 * u.deg)
)
for key in ["ra", "dec"]:
    ax.coords[key].set_ticklabel_visible(False)
    ax.coords[key].set_ticks_visible(False)
transform = ax.get_transform("world")
scalar_mappable = cm.ScalarMappable(norm=colors.LogNorm(vmin=3, vmax=200), cmap="Blues")
for cost, region in sorted(
    zip(costs * allsky_visits, fov.footprint(inscribed_circle, coords)),
    key=itemgetter(0),
):
    vertices = circle_to_polygon(region, 16).vertices
    for cut_vertices in plot.cut_prime_meridian(
        np.column_stack(((vertices.ra + shift).to_value(u.rad), vertices.dec.rad))
    ):
        cut_vertices[:, 0] -= shift.to_value(u.rad)
        ax.add_patch(
            plt.Polygon(
                np.rad2deg(cut_vertices),
                ec="none",
                lw=0,
                fc=scalar_mappable.to_rgba(cost),
                closed=True,
                fill=True,
                transform=transform,
            )
        )
for region in fov.footprint(inscribed_circle, coords):
    vertices = circle_to_polygon(region, 16).vertices
    for cut_vertices in plot.cut_prime_meridian(
        np.column_stack(((vertices.ra + shift).to_value(u.rad), vertices.dec.rad))
    ):
        cut_vertices[:, 0] -= shift.to_value(u.rad)
        ax.add_patch(
            plt.Polygon(
                np.rad2deg(cut_vertices),
                ec="black",
                lw=0.125,
                fc="none",
                closed=True,
                fill=False,
                transform=transform,
            )
        )
cbar = plt.colorbar(
    scalar_mappable,
    ax=ax,
    orientation="horizontal",
)
cbar.set_label("Number of exposures")

for tick, count in zip(*np.unique(costs, return_counts=True)):
    kwargs = dict(
        xytext=(-20, 10),
        ha="right",
        arrowprops=dict(arrowstyle="->", connectionstyle="angle,angleA=0,angleB=90"),
    )
    if tick == costs.max():
        kwargs["xytext"] = (20, 10)
        kwargs["ha"] = "left"
        kwargs["arrowprops"]["connectionstyle"] = "angle,angleA=0,angleB=90"
    cbar.ax.annotate(
        text=f"{count}\nfields",
        xy=(tick * allsky_visits, 1),
        xycoords=("data", "axes fraction"),
        textcoords="offset pixels",
        va="bottom",
        **kwargs,
    )

for region in plot_regions:
    ax.add_patch(region.to_pixel(ax.wcs).as_artist(edgecolor="orange", linewidth=1))
fig.savefig("../visualizations/costs.pdf")

Convert to a weighted graph.

In [ ]:
# Construct graph from nearest neighbors
distances = coords[:, np.newaxis].separation(coords).to_value(u.deg)
np.fill_diagonal(distances, np.inf)  # no loops
n = 2 * _geodesic.num_edges(*geodesic_args)
i = np.argpartition(distances, n, axis=None)[:n]
adjacency = np.zeros_like(distances, dtype=bool)
adjacency.flat[i] = True
graph = nx.from_numpy_array(adjacency)

# Add costs to graph
for (_, data), cost in zip(graph.nodes(data=True), costs):
    data["cost"] = cost

In [ ]:
# Construct edge weights that reward strips in right ascension.
d_lon, d_lat = coords[:, np.newaxis].spherical_offsets_to(coords[np.newaxis, :])
lon_weight = 1
lat_weight = 1000
power = 3
edge_weights = np.ceil(
    (lon_weight * np.abs(d_lon) ** power + lat_weight * np.abs(d_lat) ** power).value
).astype(int)
edge_weights[~adjacency] = 0

for i, j, data in graph.edges(data=True):
    data["ground_based_weight"] = edge_weights[i, j]

That was pretty slow too. Let's try to do this by hand.

In [ ]:
nmax = 20
nmin = nmax - 1
node_costs = graph.nodes(data="cost")
remaining_graph = graph.subgraph(fields["field_id"][~fields["in_mc"]]).copy()
partition = np.full(graph.number_of_nodes(), -1, dtype=np.intp)
partition[fields["in_mc"]] = 0

while remaining_graph.number_of_nodes() > 0:
    max_cost = max(node_costs[node] for node in remaining_graph.nodes)
    print(max_cost)
    if max_cost > 1:
        nodes, *_ = nx.connected_components(
            nx.subgraph_view(
                remaining_graph, filter_node=lambda node: node_costs[node] == max_cost
            )
        )
        remainder = len(nodes) % nmax
        if remainder != 0:
            needed = nmax - remainder
            boundary = []
            boundary_max_node_cost = max(
                node_costs[node]
                for node in nx.node_boundary(remaining_graph, [*nodes, *boundary])
            )
            while len(boundary) < needed:
                boundary.extend(
                    sorted(
                        nx.node_boundary(
                            remaining_graph,
                            [*nodes, *boundary],
                            [
                                node
                                for node in remaining_graph.nodes
                                if node not in boundary
                                and node_costs[node] == boundary_max_node_cost
                            ],
                        ),
                        key=lambda node: node_costs[node],
                        reverse=True,
                    )
                )
            nodes = [*nodes, *boundary[:needed]]
        kmin = int(np.floor(len(nodes) / nmax))
        kmax = int(np.ceil(len(nodes) / nmin))
        subgraph = remaining_graph.subgraph(nodes)
        for k in range(max(1, kmin), kmax + 1):
            for seed in [42]:
                subgraph_partition = partition_graph(
                    subgraph,
                    k,
                    seed=seed,
                    contig=True,
                    no2hop=True,
                    ncuts=1000,
                    nseps=1000,
                    niter=10000,
                    recursive=True,
                )
                _, counts = np.unique(subgraph_partition, return_counts=True)
                if np.all((counts == nmin) | (counts == nmax)):
                    break
            if np.all((counts == nmin) | (counts == nmax)):
                break
        else:
            raise RuntimeError("No suitable partition")
    else:
        # nodes, *_ = nx.connected_components(
        #     nx.subgraph_view(
        #         remaining_graph, filter_node=lambda node: node_costs[node] == max_cost
        #     )
        # )
        # subgraph = remaining_graph.subgraph(nodes)
        subgraph = remaining_graph
        kmin = int(np.floor(remaining_graph.number_of_nodes() / nmax))
        kmax = int(np.ceil(remaining_graph.number_of_nodes() / nmin))
        nodes = graph.nodes
        for k in range(max(1, kmin), kmax + 1):
            for seed in [42]:
                subgraph_partition = partition_graph(
                    subgraph,
                    k,
                    seed=seed,
                    contig=True,
                    no2hop=True,
                    ncuts=1000,
                    nseps=1000,
                    niter=10000,
                )
                _, counts = np.unique(subgraph_partition, return_counts=True)
                if np.all((counts >= nmin) & (counts <= nmax)):
                    break
            if np.all((counts >= nmin) & (counts <= nmax)):
                break
        else:
            raise RuntimeError("No suitable partition")

    part0 = partition.max() + 1
    for node, part in zip(subgraph.nodes, subgraph_partition + part0):
        partition[node] = part
    remaining_graph.remove_nodes_from(nodes)

    fig = plt.figure()
    ax = fig.add_subplot(
        projection="astro aitoff", center=SkyCoord(12 * u.hourangle - shift, 0 * u.deg)
    )
    for comp in nx.connected_components(remaining_graph):
        ax.scatter_coord(fields["target_coord"][list(comp)], s=4)
    for region in plot_regions:
        ax.add_patch(region.to_pixel(ax.wcs).as_artist(edgecolor="orange", linewidth=1))

fig = plt.figure()
ax = fig.add_subplot(projection="astro aitoff", center="8h 0d")
ax.scatter_coord(
    coords,
    c=partition_graph_color(graph, partition)[partition],
    s=2,
    cmap="cool",
)
for region in plot_regions:
    ax.add_patch(region.to_pixel(ax.wcs).as_artist(color="black"))

_, counts = np.unique(partition, return_counts=True)
np.unique(counts, return_counts=True)

In [ ]:
fig = plt.figure()
shift = 4 * u.hourangle
ax = plt.axes(
    projection="astro aitoff", center=SkyCoord(12 * u.hourangle - shift, 0 * u.deg)
)
for key in ["ra", "dec"]:
    ax.coords[key].set_ticklabel_visible(False)
    ax.coords[key].set_ticks_visible(False)
transform = ax.get_transform("world")
graph_color = partition_graph_color(graph, partition)[partition]
scalar_mappable = cm.ScalarMappable(cmap="cool")
scalar_mappable.set_array(graph_color)
for c, region in zip(graph_color, fov.footprint(inscribed_circle, coords)):
    vertices = circle_to_polygon(region, 16).vertices
    for cut_vertices in plot.cut_prime_meridian(
        np.column_stack(((vertices.ra + shift).to_value(u.rad), vertices.dec.rad))
    ):
        cut_vertices[:, 0] -= shift.to_value(u.rad)
        ax.add_patch(
            plt.Polygon(
                np.rad2deg(cut_vertices),
                ec="none",
                lw=0,
                fc=scalar_mappable.to_rgba(c),
                closed=True,
                fill=True,
                transform=transform,
            )
        )
for region in fov.footprint(inscribed_circle, coords):
    vertices = circle_to_polygon(region, 16).vertices
    for cut_vertices in plot.cut_prime_meridian(
        np.column_stack(((vertices.ra + shift).to_value(u.rad), vertices.dec.rad))
    ):
        cut_vertices[:, 0] -= shift.to_value(u.rad)
        ax.add_patch(
            plt.Polygon(
                np.rad2deg(cut_vertices),
                ec="black",
                lw=0.125,
                fc="none",
                closed=True,
                fill=False,
                transform=transform,
            )
        )

for region in plot_regions:
    ax.add_patch(region.to_pixel(ax.wcs).as_artist(edgecolor="orange", linewidth=1))
fig.savefig("../visualizations/skyblocks.pdf")

In [ ]:
fields["block_id"] = partition
fields

In [ ]:
fields[
    "field_id",
    "block_id",
    "target_coord",
    *(f"in_{key}" for key in regions.keys()),
    *(f"overlap_{key}" for key in regions.keys()),
].write("../fields.ecsv", overwrite=True)